# Práctica 1 · Clasificación: ¿este incidente es real?
**Analítica de procesos para ciberseguridad**

**El problema.** En un centro de operaciones de seguridad (SOC) llegan miles de alertas. Los detectores avisan de casi todo, y la mayoría de los avisos **no son ataques**. Un analista tiene que revisar cada incidente y decidir qué hacer con él. A esa decisión se le llama **triaje**, como en urgencias: ¿es una **amenaza real**, **no hay amenaza**, o es **actividad esperada**?

**La tarea.** Vamos a enseñarle a un modelo a tomar esa decisión viendo incidentes que analistas reales ya clasificaron, y a **medir cuánto se equivoca**. Es **clasificación**: dada la información de un incidente, elegir una de varias etiquetas.

**Las tres etiquetas** (las pone el analista del cliente, y es la respuesta correcta que tenemos):

| Etiqueta | Qué significa | En urgencias sería... |
|---|---|---|
| **Amenaza real** | Es un ataque de verdad | Un paciente grave |
| **Sin amenaza** | La alerta sonó, pero no pasaba nada malo | Una falsa alarma |
| **Actividad esperada** | Sí pasó algo, pero es normal o autorizado (por ejemplo, una prueba de penetración o un administrador legítimo) | Un paciente que llegó, pero no es una urgencia |

> **Sobre los nombres.** En un SOC estas tres etiquetas se llaman *verdadero positivo*, *falso positivo* y *positivo benigno*. Aquí usamos otros nombres a propósito: así no se confunden con los **errores del modelo** (los falsos positivos y los falsos negativos de la matriz de confusión), que aparecerán más abajo.

**Los datos.** Incidentes reales de **Microsoft** (conjunto GUIDE, Kaggle, licencia CDLA-Permissive-2.0): clientes de todo el mundo, con identificadores anonimizados. Cada incidente agrupa una o más alertas de seguridad.

> **Fuente y cambios.** Freitas, Kalajdjieski, Gharib y McCann (2025). *AI-Driven Guided Response for Security Operation Centers with Microsoft Copilot for Security.* ACM Web Conference 2025. Cambios hechos con fines académicos: el original trae una fila por cada evidencia de cada alerta (más de 4 millones de filas); aquí se resumió en **una fila por incidente**, se tomó una muestra de 80.000 incidentes, y los nombres de columnas y etiquetas pasaron a español (script `scripts/preparar_guide.py`).

**Qué vas a hacer**
1. Mirar los datos y entender qué describe cada variable.
2. Ver qué variables se parecen más a la etiqueta (antes de modelar).
3. Probar seis modelos y comparar cuánto aciertan **y en qué se equivocan**.
4. Ver por qué el mismo modelo puede acertar entre 60 % y 90 % según **cómo lo midas**.

## 0 · Preparación

In [ ]:
# Importa las librerías: pandas (tablas), numpy (cálculos) y matplotlib (gráficos)
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")                # oculta advertencias para que la salida se lea limpia
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)   # tablas más anchas

# Dirección de los datos: repositorio de GitHub y versión fija de donde se descargan los archivos
REPO = "lsandinop/analitica-procesos-ciberseguridad"
VERSION = "main"
DATA_BASE = "" if REPO.startswith("USUARIO") else f"https://raw.githubusercontent.com/{REPO}/{VERSION}/datos/"

def cargar(nombre):
    # Lee un archivo CSV y lo devuelve como tabla (DataFrame). Lo busca primero en GitHub;
    # si no está disponible, en la carpeta local.
    if DATA_BASE:
        try:
            return pd.read_csv(DATA_BASE + nombre)
        except Exception:
            print("No pude descargar el archivo de GitHub; busco una copia local.")
    for ruta in (nombre, "datos/" + nombre, "../datos/" + nombre, "../../datos/" + nombre):
        if os.path.exists(ruta):
            return pd.read_csv(ruta)
    raise FileNotFoundError(f"No encuentro {nombre}: súbelo a la carpeta del cuaderno")

incidentes = cargar("guide_incidentes.csv")     # una fila por incidente de SOC, con la etiqueta del analista en `grado`
print(f"{len(incidentes):,} incidentes de {incidentes.organizacion.nunique():,} organizaciones · {incidentes.shape[1]} columnas".replace(",", "."))

**Una mirada a los datos.** Cada fila es un incidente. Aquí van los primeros tres, de lado para que se lean todas las columnas:

In [ ]:
incidentes.head(3).T          # .T pone las columnas como filas, para leer cada incidente de arriba hacia abajo

**Qué describe cada variable.** Un incidente es un grupo de alertas relacionadas. Las columnas lo resumen así:

| Grupo | Variables | Qué cuentan |
|---|---|---|
| **Etiqueta** | `grado` | La respuesta correcta: lo que decidió el analista |
| **Cliente** | `organizacion` | Identificador anónimo de la organización |
| **Tamaño** | `n_alertas`, `n_evidencias`, `n_detectores`, `n_tacticas`, `n_paises` | Cuántas alertas, pruebas, detectores distintos, etapas del ataque y países hay en el incidente |
| **Tiempo** | `minutos_duracion`, `hora_inicio`, `dia_semana` | Cuánto duró y cuándo empezó (hora 0-23; día 0 = lunes) |
| **Etapa del ataque** | `tactica_acceso_inicial`, `tactica_exfiltracion`, ... | Qué proporción de las alertas corresponde a cada **táctica** del catálogo MITRE ATT&CK (acceso inicial, exfiltración, movimiento lateral...) |
| **Qué se vio involucrado** | `entidad_ip`, `entidad_usuario`, `entidad_equipo`, ... | Qué proporción de las pruebas son direcciones IP, usuarios, equipos, correos, archivos... |
| **Veredictos automáticos** | `prop_veredicto_malicioso`, `prop_veredicto_sin_amenazas`, `prop_nivel_sospecha`, `prop_con_tecnica_mitre` | Qué dijeron las herramientas automáticas antes de que lo viera el analista |
| **Detector** | `detector_principal` | Identificador del detector que más alertas generó (−1 si es poco común) |

## 1 · La respuesta correcta y la línea base
La columna `grado` es nuestra **tabla preclasificada**: analistas de seguridad de las organizaciones clientes revisaron cada incidente y le pusieron una de las tres etiquetas. Eso nos permite **calificar** cualquier modelo: comparamos su respuesta contra la del analista.

¿Cuántos incidentes hay de cada tipo?

In [ ]:
conteo = incidentes.grado.value_counts()
colores = {"amenaza real": "#d9534f", "sin amenaza": "#5b9bd5", "actividad esperada": "#a8b0b6"}
ax = conteo.plot.barh(figsize=(7, 2.6), color=[colores[g] for g in conteo.index])
for i, v in enumerate(conteo): ax.text(v, i, f"  {v:,} ({v / len(incidentes):.1%})".replace(",", "."), va="center")
ax.set_xlim(0, conteo.max() * 1.3); ax.invert_yaxis(); plt.title("Incidentes por etiqueta del analista"); plt.tight_layout(); plt.show()

**La línea base.** Un «modelo» que ni mira el incidente y responde siempre la etiqueta más común (*actividad esperada*) acierta el **49,2 %**. No sirve para nada, pero es la **vara mínima**: un modelo que no la supere no aprendió nada. Y fíjate que, con tres etiquetas, **acertar al azar es más difícil** que en una decisión de sí o no.

## 2 · Separar para entrenar y para probar
Igual que cuando se estudia para un examen: el modelo **aprende** con el 70 % de los incidentes y lo **calificamos** con el 30 % restante, que nunca vio. La semilla fija el sorteo para que todos tengamos los mismos grupos, y `stratify` mantiene la proporción de cada etiqueta.

In [ ]:
from sklearn.model_selection import train_test_split

CLASES = ["amenaza real", "sin amenaza", "actividad esperada"]
# Variables que usa el modelo: todas menos la etiqueta y la organización (la organización la usaremos más abajo, aparte)
VARIABLES = [c for c in incidentes.columns if c not in ("grado", "organizacion")]
X, y = incidentes[VARIABLES], incidentes.grado

# Sortea qué incidentes van a entrenamiento (70 %) y cuáles a prueba (30 %), manteniendo la proporción de cada etiqueta
pos_train, pos_test = train_test_split(np.arange(len(incidentes)), test_size=0.3, stratify=y, random_state=123456789)
X_train, X_test, y_train, y_test = X.iloc[pos_train], X.iloc[pos_test], y.iloc[pos_train], y.iloc[pos_test]
print(f"{len(X_train):,} incidentes para entrenar · {len(X_test):,} para probar".replace(",", "."))

# ¿Se mantuvo la proporción de cada etiqueta?
prop = pd.DataFrame({"conjunto completo": y.value_counts(normalize=True), "entrenamiento (70 %)": y_train.value_counts(normalize=True),
                     "prueba (30 %)": y_test.value_counts(normalize=True)}).loc[CLASES] * 100
print("\n% de cada etiqueta por grupo:"); print(prop.round(1))

base = (y_test == "actividad esperada").mean()
print(f"\nLínea base: responder siempre 'actividad esperada' acierta el {base:.1%} de la prueba.")

**Una función para calificar.** Cada modelo entrega una etiqueta por incidente. Esta función la compara con la del analista, dibuja la **matriz de confusión** (filas: lo que dijo el analista; columnas: lo que dijo el modelo) y guarda una fila en una tabla de resultados.

Con tres etiquetas miramos tres cosas: la **exactitud** (cuántos acertó), el ***recall*** de cada etiqueta (de los incidentes de ese tipo, cuántos encontró) y el **F1 promedio** (una sola cifra que mezcla precisión y *recall* de las tres etiquetas por igual; es más exigente que la exactitud cuando las etiquetas están desbalanceadas).

In [ ]:
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

resultados = []     # aquí se acumula una fila por cada modelo que calificamos

def evaluar(nombre, real, pred, mostrar=True):
    # Califica un modelo: exactitud, F1 promedio y recall de cada etiqueta; guarda la fila y dibuja la matriz de confusión
    real, pred = np.asarray(real), np.asarray(pred)
    cm = confusion_matrix(real, pred, labels=CLASES)
    recall = cm.diagonal() / cm.sum(axis=1)                      # de los incidentes de cada tipo, ¿qué fracción encontró?
    fila = {"modelo": nombre, "exactitud": accuracy_score(real, pred), "F1 promedio": f1_score(real, pred, average="macro"),
            **{"recall " + c: r for c, r in zip(CLASES, recall)}}
    resultados[:] = [r for r in resultados if r["modelo"] != nombre] + [fila]      # si se repite el modelo, reemplaza la fila
    if mostrar:
        fig, ax = plt.subplots(figsize=(4.8, 3.6))
        ax.imshow(cm, cmap="Blues")
        for (i, j), v in np.ndenumerate(cm):
            ax.text(j, i, f"{v:,}".replace(",", "."), ha="center", va="center", color="white" if v > cm.max() / 2 else "black")
        ax.set_xticks(range(3), [c.replace(" ", "\n") for c in CLASES], fontsize=8); ax.set_yticks(range(3), [c.replace(" ", "\n") for c in CLASES], fontsize=8)
        ax.set_xlabel("Lo que dijo el modelo"); ax.set_ylabel("Lo que dijo el analista"); ax.set_title(nombre, fontsize=9)
        plt.tight_layout(); plt.show()
        print(f"Exactitud {fila['exactitud']:.1%} · F1 promedio {fila['F1 promedio']:.2f} · encuentra "
              f"{recall[0]:.0%} de las amenazas reales, {recall[1]:.0%} de los incidentes sin amenaza y {recall[2]:.0%} de los de actividad esperada")

def tabla_resultados():
    # La tabla acumulada de todos los modelos calificados hasta ahora
    return pd.DataFrame(resultados).set_index("modelo").round(3)

def ver_predicciones(pred, n_errores=6, n_aciertos=3):
    # Incidentes de la prueba con la respuesta del analista, la del modelo y si acertó (errores primero)
    t = X_test.assign(analista=y_test.values, modelo=np.asarray(pred))
    t["veredicto"] = np.where(t.analista == t.modelo, "✓ acierta", "✗ se equivoca")
    t["etapa principal"] = incidentes.loc[X_test.index, [c for c in incidentes.columns if c.startswith("tactica_")]].idxmax(axis=1).str.replace("tactica_", "").values
    sel = pd.concat([t[t.veredicto.str.startswith("✗")].head(n_errores), t[t.veredicto.str.startswith("✓")].head(n_aciertos)])
    return sel[["analista", "modelo", "veredicto", "n_alertas", "etapa principal", "prop_veredicto_malicioso", "minutos_duracion"]]

## 3 · Antes de modelar: ¿qué variables se relacionan con la etiqueta?
Un modelo aprende relaciones entre las **variables** y el **objetivo** (la etiqueta). Conviene mirarlas antes, con los datos de **entrenamiento** (la prueba no se toca para explorar).

**Cómo leer los gráficos.** Cada uno compara las **tres etiquetas** con **una** variable. La pregunta de fondo es siempre la misma: *¿esta variable se comporta distinto según la etiqueta?* Si sí, nos ayuda a distinguirlas; si las tres barras se parecen, no ayuda.

| Gráfico | Qué se mide | Cómo se lee |
|---|---|---|
| **(a) Veredicto automático** | Antes de que el analista lo viera, las herramientas automáticas marcaron algunas alertas como «maliciosas» | Cada barra es el **% de alertas marcadas «maliciosas»**, en promedio, entre los incidentes de esa etiqueta. Una barra de 5 % significa que en esos incidentes, en promedio, 5 de cada 100 alertas ya venían marcadas así |
| **(b) Una sola alerta** | Cuántas alertas trae el incidente | Cada barra es el **% de incidentes de esa etiqueta que tienen una única alerta** (de cada 100 incidentes, cuántos) |
| **(c) Etapa del ataque** | Cada alerta pertenece a una etapa del ataque (táctica de MITRE ATT&CK). De cada incidente tomamos la etapa que más pesa | Cada **columna** es una etiqueta y suma 100 %: dentro de ella, qué % de los incidentes tiene esa etapa como principal. **Más oscuro = más frecuente** |

In [ ]:
ent = incidentes.iloc[pos_train]
nombres = [c.replace(" ", "\n") for c in CLASES]            # nombres en dos líneas para que quepan bajo las barras
fig, ejes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1, 1, 1.25]})

# (a) ¿Qué dijeron las herramientas automáticas? Por etiqueta, el % promedio de alertas con veredicto «malicioso»
m = ent.groupby("grado").prop_veredicto_malicioso.mean().loc[CLASES] * 100
barras = ejes[0].bar(nombres, m, color=[colores[c] for c in CLASES]); ejes[0].bar_label(barras, fmt="%.1f %%", fontsize=10)
ejes[0].set_title("(a) ¿Las herramientas automáticas\nya decían «malicioso»?", fontsize=10)
ejes[0].set_ylabel("% de alertas marcadas «maliciosas»\n(promedio de los incidentes de cada etiqueta)", fontsize=8); ejes[0].set_ylim(0, m.max() * 1.3)

# (b) ¿Cuántas alertas tiene el incidente? Por etiqueta, el % de incidentes con una sola alerta
u = ent.assign(una=ent.n_alertas == 1).groupby("grado").una.mean().loc[CLASES] * 100
barras = ejes[1].bar(nombres, u, color=[colores[c] for c in CLASES]); ejes[1].bar_label(barras, fmt="%.0f %%", fontsize=10)
ejes[1].set_title("(b) ¿El incidente tiene\nuna sola alerta?", fontsize=10)
ejes[1].set_ylabel("% de los incidentes de cada etiqueta", fontsize=8); ejes[1].set_ylim(0, 100)

# (c) ¿En qué etapa del ataque está? La etapa (táctica) con más peso en cada incidente; cada columna (etiqueta) suma 100 %
tacticas = [c for c in ent.columns if c.startswith("tactica_")]
principal = ent[tacticas].idxmax(axis=1).str.replace("tactica_", "").str.replace("_", " ")      # la táctica con más peso en cada incidente
top = list(principal.value_counts().head(5).index)                                                  # las 5 etapas más comunes; el resto va en «otras»
tabla = pd.crosstab(principal.where(principal.isin(top), "otras"), ent.grado, normalize="columns")[CLASES].loc[top + ["otras"]] * 100
ejes[2].imshow(tabla.values, cmap="Blues", vmin=0, vmax=70, aspect="auto")
for (i, j), v in np.ndenumerate(tabla.values):
    ejes[2].text(j, i, f"{v:.0f} %", ha="center", va="center", fontsize=9, color="white" if v > 40 else "black")
ejes[2].set_xticks(range(3), nombres, fontsize=9)
ejes[2].set_yticks(range(len(tabla)), [t.replace("exfiltracion", "exfiltración").replace("ejecucion", "ejecución") for t in tabla.index], fontsize=9)
ejes[2].set_title("(c) ¿En qué etapa del ataque está?\n(cada columna suma 100 %)", fontsize=10)
plt.tight_layout(); plt.show()

**Qué leer ahí:**
- **(a)** Las herramientas automáticas casi no opinan: en el 88 % de los incidentes ninguna dio un veredicto. Cuando opinan, las amenazas reales tienen algo más de alertas «maliciosas» (4,9 %) que la actividad esperada (4,0 %) y bastante más que los incidentes sin amenaza (1,4 %). **Ayuda, pero no separa.**
- **(b)** Casi todos los incidentes tienen **una sola alerta**, sea cual sea la etiqueta (entre 86 % y 91 %). **Esta variable no distingue** una etiqueta de otra.
- **(c)** La mezcla de etapas sí cambia entre columnas: en «sin amenaza» pesa el «acceso inicial» (58 %); en «amenaza real» pesa más la «actividad sospechosa» (31 %) y la «exfiltración» casi no aparece (4 %), mientras que en «actividad esperada» llega al 18 %. **Da pistas, pero ninguna etapa decide sola.**

**Ninguna variable separa las tres etiquetas por sí sola**: en los gráficos las diferencias son pequeñas o se mezclan. Por eso se usa un modelo, que combina muchas variables a la vez.

## 4 · Seis modelos, del más explicable al más potente
Probamos los modelos más usados con datos como estos. Todos ven **las mismas variables** y se califican con **los mismos incidentes de prueba**.

### Modelo 1 · Árbol de decisión de 3 niveles
Un árbol hace preguntas sí/no sobre las variables y sigue un camino hasta una respuesta. Con 3 niveles cabe en una pantalla y **se puede leer entero**:

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

arbol3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)     # a lo sumo 3 preguntas seguidas
plt.figure(figsize=(17, 7))
plot_tree(arbol3, feature_names=VARIABLES, class_names=[c.replace(" ", "\n") for c in arbol3.classes_], filled=True, fontsize=8, impurity=False)
plt.title("Árbol de 3 niveles: cada cuadro es una pregunta; se va a la izquierda si la respuesta es «sí»"); plt.show()

In [ ]:
pred_arbol3 = arbol3.predict(X_test)
evaluar("1 · Árbol de 3 niveles", y_test, pred_arbol3)

**Lee los errores:** incidentes de la prueba con lo que dijo el analista, lo que dijo el modelo y si acertó.

In [ ]:
ver_predicciones(pred_arbol3)

### Modelo 2 · Árbol de 8 niveles
Más preguntas separan mejor, pero se pierde la lectura fácil. Compara cuánto acierta con los incidentes con los que **aprendió** y con los que **nunca vio**:

In [ ]:
arbol8 = DecisionTreeClassifier(max_depth=8, random_state=0).fit(X_train, y_train)
print(f"Acierta {arbol8.score(X_train, y_train):.1%} de los incidentes con que aprendió y {arbol8.score(X_test, y_test):.1%} de los que no vio")
pred_arbol8 = arbol8.predict(X_test)
evaluar("2 · Árbol de 8 niveles", y_test, pred_arbol8, mostrar=False)
tabla_resultados()

### Modelo 3 · Regresión logística
Le da **un peso a cada variable** (positivo o negativo), suma los pesos y convierte la suma en una probabilidad para cada etiqueta. Es muy explicable, pero solo capta relaciones «en línea recta». Antes se **estandarizan** las variables (se pasan a una misma escala) para que pesos y variables sean comparables.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

escala = StandardScaler().fit(X_train)                                  # media 0 y desviación 1, calculadas SOLO con entrenamiento
Xe_train, Xe_test = escala.transform(X_train), escala.transform(X_test)
logistica = LogisticRegression(max_iter=3000).fit(Xe_train, y_train)
pred_logistica = logistica.predict(Xe_test)
evaluar("3 · Regresión logística", y_test, pred_logistica, mostrar=False)
tabla_resultados()

### Modelo 4 · Bosque aleatorio
Muchos árboles (aquí 200), cada uno entrenado con una parte distinta de los datos, que **votan** la respuesta. Un árbol solo se equivoca de formas distintas a los otros, y el voto compensa. A cambio, ya no se puede leer: son 200 árboles.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

bosque = RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=0).fit(X_train, y_train)
pred_bosque = bosque.predict(X_test)
evaluar("4 · Bosque aleatorio", y_test, pred_bosque, mostrar=False)
tabla_resultados()

### Modelo 5 · *Gradient boosting*
En vez de árboles independientes, se entrena un árbol pequeño, luego otro **que corrige los errores del anterior**, y así muchas veces. Hoy es lo que más se usa con tablas de datos como esta.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

boosting = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)
pred_boosting = boosting.predict(X_test)
evaluar("5 · Gradient boosting", y_test, pred_boosting)

### Modelo 6 · Red neuronal
Capas de «neuronas» que combinan las variables y se ajustan solas. Vocabulario mínimo para leer el código:

| Término | Qué es | Si lo subes |
|---|---|---|
| **Capa oculta** | Una etapa que transforma lo que recibe. Aquí: una de 64 neuronas y otra de 32 | Más capas o neuronas: más capacidad, y más riesgo de memorizar |
| **Época** | Una pasada completa por todos los incidentes de entrenamiento | Pocas: no alcanza a aprender. Muchas: memoriza |
| **Estandarizar** | Pasar las variables a una escala común | Sin esto, una red aprende mal |

In [ ]:
from sklearn.neural_network import MLPClassifier

red = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=60, random_state=0).fit(Xe_train, y_train)     # 60 épocas como máximo
pred_red = red.predict(Xe_test)
evaluar("6 · Red neuronal", y_test, pred_red, mostrar=False)
print("Parámetros que aprendió la red:", sum(w.size for w in red.coefs_) + sum(b.size for b in red.intercepts_))
tabla_resultados()

In [ ]:
# Comparación de los seis modelos: exactitud y F1 promedio (la línea base es 49,2 %)
t = tabla_resultados()[["exactitud", "F1 promedio"]]
ax = t.iloc[::-1].plot.barh(figsize=(9, 4), color=["#5b9bd5", "#f0ad4e"]); ax.set_xlim(0, 1); ax.legend(loc="lower right")
plt.title("Los seis modelos con los mismos incidentes de prueba"); plt.tight_layout(); plt.show()

**Qué mirar.** Los modelos de conjunto (bosque, *boosting*) y la red acertan más que los árboles pequeños y que la regresión logística, pero **ninguno es perfecto**: el triaje es difícil incluso para un analista. Y el que más acierta es el que menos se puede explicar. Esa es la tensión de todo este curso: **precisión contra explicabilidad**.

**¿Qué variables usó el bosque?** Una forma de «abrir la caja»: cuánto contribuye cada variable a las decisiones del bosque.

In [ ]:
imp = pd.Series(bosque.feature_importances_, index=VARIABLES).sort_values().tail(12)
imp.plot.barh(figsize=(7, 4.2), color="#5b9bd5"); plt.xlabel("importancia según el bosque aleatorio"); plt.title("Las 12 variables que más pesaron"); plt.tight_layout(); plt.show()

## 5 · La medición honesta: ¿para quién clasifica el modelo?
Todas las cifras de arriba sirven para **un caso**: el modelo clasifica incidentes de organizaciones que **ya vio** durante el entrenamiento. Pero ¿qué pasa si llega un **cliente nuevo**? Repitamos con el mismo modelo (*gradient boosting*), cambiando solo **cómo se reparten** los incidentes:

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, KFold

org = incidentes.organizacion

# (A) Organizaciones nuevas: todos los incidentes de una organización van a UN solo grupo, así el modelo se prueba con clientes que no conoce
tr_o, te_o = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=123456789).split(X, y, org))
modelo_o = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[tr_o], y.iloc[tr_o])
pred_o = modelo_o.predict(X.iloc[te_o])
print(f"Clientes nuevos: exactitud {accuracy_score(y.iloc[te_o], pred_o):.1%} · F1 promedio {f1_score(y.iloc[te_o], pred_o, average='macro'):.2f}")
print(f"(al azar, con clientes ya vistos: exactitud {accuracy_score(y_test, pred_boosting):.1%} · F1 promedio {f1_score(y_test, pred_boosting, average='macro'):.2f})")

**Y si el modelo conoce a cada cliente.** Un SOC aprende de sus propias decisiones pasadas. Le agregamos una variable nueva: **el historial de la organización**, es decir, la proporción de cada etiqueta en *sus* incidentes de entrenamiento. Para que no «copie» la respuesta, el historial de cada incidente de entrenamiento se calcula **sin mirar a ese incidente** (validación cruzada de 5 partes).

In [ ]:
def historial_organizacion(idx_aprender, idx_aplicar, suavizado=20):
    # Para cada incidente de idx_aplicar: la proporción de cada etiqueta en los incidentes de SU organización que están en idx_aprender.
    # El suavizado acerca a la proporción global a las organizaciones con pocos incidentes.
    Y = pd.get_dummies(y.iloc[idx_aprender])[CLASES].astype(float); Y["o"] = org.iloc[idx_aprender].values
    global_ = Y[CLASES].mean(); suma = Y.groupby("o")[CLASES].sum(); n = Y.groupby("o").size()
    prop_org = suma.add(global_ * suavizado, axis=1).div(n + suavizado, axis=0)
    return pd.DataFrame(prop_org.reindex(org.iloc[idx_aplicar].values).fillna(global_).values,
                        columns=["historial_" + c.replace(" ", "_") for c in CLASES], index=X.index[idx_aplicar])

hist_train = pd.concat([historial_organizacion(pos_train[a], pos_train[b]) for a, b in KFold(5, shuffle=True, random_state=0).split(pos_train)])
hist_test = historial_organizacion(pos_train, pos_test)         # la prueba usa el historial de entrenamiento completo
modelo_h = HistGradientBoostingClassifier(random_state=0).fit(X_train.join(hist_train), y_train)
pred_h = modelo_h.predict(X_test.join(hist_test))
evaluar("7 · Gradient boosting con historial de la organización", y_test, pred_h)

In [ ]:
# Las tres formas de medir el MISMO tipo de modelo, lado a lado
comparacion = pd.Series({"Línea base (siempre 'actividad esperada')": base,
                         "Clientes NUEVOS": accuracy_score(y.iloc[te_o], pred_o),
                         "Clientes ya vistos": accuracy_score(y_test, pred_boosting),
                         "Conociendo el historial del cliente": accuracy_score(y_test, pred_h)})
ax = comparacion.iloc[::-1].plot.barh(figsize=(8, 3.2), color=["#a8b0b6", "#d9534f", "#f0ad4e", "#5cb85c"][::-1]); ax.set_xlim(0, 1)
for i, v in enumerate(comparacion.iloc[::-1]): ax.text(v + 0.01, i, f"{v:.0%}", va="center")
plt.title("Exactitud del mismo modelo según cómo se mide"); plt.tight_layout(); plt.show()

**La lección.** Lo que es «actividad esperada» **lo decide cada organización**: una prueba de penetración autorizada es normal para un cliente y alarmante para otro. Un modelo que conoce al cliente acierta mucho; con un cliente nuevo cae a 60 %, solo unos 11 puntos por encima de la línea base. **Una cifra de exactitud no significa nada si no sabes con qué datos se midió.** Es la misma trampa de siempre (partir los datos al azar), ahora con organizaciones.

## 6 · El costo de equivocarse: no todos los errores pesan igual
En un SOC, **no detectar una amenaza real** es mucho peor que revisar de más un incidente *sin amenaza*. El modelo entrega, para cada incidente, una **probabilidad** de ser amenaza real. Si en vez de elegir la etiqueta más probable decidimos «revisar todo incidente con probabilidad mayor a un **umbral**», podemos cambiar cuántas amenazas reales encontramos y cuántos incidentes de más aceptamos revisar:

In [ ]:
prob_amenaza = boosting.predict_proba(X_test)[:, list(boosting.classes_).index("amenaza real")]    # probabilidad de ser una amenaza real
es_amenaza = (y_test == "amenaza real").values
filas = []
for umbral in (0.5, 0.4, 0.3, 0.2, 0.1):
    marcados = prob_amenaza >= umbral
    filas.append({"umbral": umbral, "incidentes a revisar": int(marcados.sum()), "% del total": marcados.mean() * 100,
                  "amenazas reales encontradas": int((marcados & es_amenaza).sum()), "% de las amenazas reales": (marcados & es_amenaza).sum() / es_amenaza.sum() * 100,
                  "% de lo revisado que era amenaza real": (marcados & es_amenaza).sum() / max(marcados.sum(), 1) * 100})
pd.DataFrame(filas).set_index("umbral").round(1)

**Cómo leerlo.** Bajar el umbral encuentra más amenazas reales, pero obliga a revisar muchos más incidentes (más trabajo y más **fatiga de alertas**). No hay un umbral «correcto»: depende de cuántos incidentes puede revisar el equipo y de cuánto cuesta dejar pasar una amenaza real. **Esa decisión es de negocio, no del modelo.**

## Cierre
- **Clasificación** = aprender de ejemplos ya etiquetados a elegir una etiqueta.
- Las **variables** importan: se miran antes de modelar, y el modelo las combina.
- Los modelos más potentes acertaron más, pero se explican menos.
- **Cómo mides decide lo que crees saber:** clientes nuevos o ya vistos, al azar o por organización.
- Acertar no basta: hay que mirar **qué** errores comete y **cuánto cuesta** cada uno.